# Molecular Devices SpectraMax

The SpectraMax family of plate readers from Molecular Devices communicate over RS-232 serial. Both models share the same serial driver ({class}`~pylabrobot.molecular_devices.spectramax.backend.MolecularDevicesDriver`), but differ in capabilities.

| Model | PLR Name | Absorbance | Fluorescence | Luminescence | Temperature Control |
|---|---|---|---|---|---|
| SpectraMax M5 | `SpectraMaxM5` | yes | yes | yes | yes |
| SpectraMax 384 Plus | `SpectraMax384Plus` | yes | no | no | yes |

Connect the reader to your computer via an RS-232 serial cable (or USB-to-serial adapter). Note the serial port name (e.g. `COM3` on Windows, `/dev/ttyUSB0` on Linux).

## Setup

In [ ]:
from pylabrobot.molecular_devices.spectramax import SpectraMaxM5

reader = SpectraMaxM5(name="spectramax", port="/dev/ttyUSB0")  # replace with your port
await reader.setup()

For the SpectraMax 384 Plus:

In [ ]:
# from pylabrobot.molecular_devices.spectramax import SpectraMax384Plus
#
# reader = SpectraMax384Plus(name="spectramax", port="/dev/ttyUSB0")
# await reader.setup()

Assign a plate to the reader's built-in plate holder:

In [ ]:
from pylabrobot.resources import Cor_96_wellplate_360ul_Fb

plate = Cor_96_wellplate_360ul_Fb(name="plate")
reader.plate_holder.assign_child_resource(plate)

## Absorbance

Both the M5 and 384 Plus support absorbance reads. For the full API, see [Absorbance](../../capabilities/absorbance).

Use {class}`~pylabrobot.molecular_devices.spectramax.backend.MolecularDevicesAbsorbanceBackend.AbsorbanceParams` to configure backend-specific settings like speed reads, path check, and kinetic reads.

In [ ]:
results = await reader.absorbance.read_absorbance(plate, wavelength=450)

With backend params:

In [ ]:
from pylabrobot.molecular_devices.spectramax.backend import (
    Calibrate,
    MolecularDevicesAbsorbanceBackend,
)

results = await reader.absorbance.read_absorbance(
    plate,
    wavelength=450,
    backend_params=MolecularDevicesAbsorbanceBackend.AbsorbanceParams(
        speed_read=True,
        calibrate=Calibrate.ONCE,
    ),
)

## Fluorescence (M5 only)

The SpectraMax M5 supports fluorescence reads. For the full API, see [Fluorescence](../../capabilities/fluorescence).

Use {class}`~pylabrobot.molecular_devices.spectramax.spectramax_m5.SpectraMaxM5FluorescenceBackend.FluorescenceParams` to configure excitation/emission wavelengths, cutoff filters, PMT gain, and other settings.

In [ ]:
results = await reader.fluorescence.read_fluorescence(
    plate,
    excitation_wavelength=485,
    emission_wavelength=520,
    focal_height=0.0,
)

With backend params for PMT gain and bottom-read:

In [ ]:
from pylabrobot.molecular_devices.spectramax.spectramax_m5 import SpectraMaxM5FluorescenceBackend
from pylabrobot.molecular_devices.spectramax.backend import PmtGain

results = await reader.fluorescence.read_fluorescence(
    plate,
    excitation_wavelength=485,
    emission_wavelength=520,
    focal_height=0.0,
    backend_params=SpectraMaxM5FluorescenceBackend.FluorescenceParams(
        pmt_gain=PmtGain.HIGH,
        read_from_bottom=True,
    ),
)

## Luminescence (M5 only)

The SpectraMax M5 supports luminescence reads. For the full API, see [Luminescence](../../capabilities/luminescence).

Use {class}`~pylabrobot.molecular_devices.spectramax.spectramax_m5.SpectraMaxM5LuminescenceBackend.LuminescenceParams` to configure emission wavelengths, PMT gain, and other settings. Note that `emission_wavelengths` is required for luminescence reads on the M5.

In [ ]:
from pylabrobot.molecular_devices.spectramax.spectramax_m5 import SpectraMaxM5LuminescenceBackend

results = await reader.luminescence.read_luminescence(
    plate,
    focal_height=0.0,
    backend_params=SpectraMaxM5LuminescenceBackend.LuminescenceParams(
        emission_wavelengths=[460],
    ),
)

## Temperature control

Both models expose a {class}`~pylabrobot.capabilities.temperature_controlling.temperature_controller.TemperatureController` on `reader.tc`. Temperature range is 0--45 C. For the full API, see [Temperature Control](../../capabilities/temperature-control).

In [ ]:
await reader.tc.set_temperature(37.0)

In [ ]:
current = await reader.tc.request_temperature()
print(f"{current:.1f} \u00b0C")

In [ ]:
await reader.tc.deactivate()

## Tray control

Open and close the plate tray via the driver:

In [ ]:
await reader.driver.open()
# load plate
await reader.driver.close()

## Teardown

In [ ]:
await reader.stop()